# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Continuous-time generative models traditionally relied on stochastic diffusion equations (SDEs) where scores are learned to reverse Brownian motion. Lipman et al. (2022) introduced **Flow Matching (FM)**, proving that one can directly regress vector fields of continuous normalizing flows via simulation-free objectives. Concurrently, Liu et al. (2022) developed **Rectified Flow (RF)**, establishing that iterative reflowing straightens generative ODE trajectories into constant-speed lines. Song et al. (2023) introduced **Consistency Models (CM)**, enforcing self-consistency along probability flow trajectories to enable single-step generation without adversarial training.

### Purpose & Target Architecture
This notebook builds from first principles:
1. Straight-line Conditional Flow Matching (CFM) with target velocity $u_t = x_1 - x_0$.
2. Time-conditioned velocity prediction network $v_\theta(x, t)$.
3. Rectified Flow Euler ODE numerical simulation.
4. Consistency model parameterization enforcing exact identity boundary conditions at $t = \epsilon$.

# 2. Mathematical Foundations & Governing Equations

### Continuous Normalizing Flows and Continuity Equation
Let $p_t(x)$ denote a probability density path on $t \in [0, 1]$ generated by velocity field $v_t(x)$:
$$\frac{\partial p_t(x)}{\partial t} + \nabla \cdot (p_t(x) v_t(x)) = 0$$

### Straight-Path Conditional Flow Matching (CFM)
Using optimal transport straight paths between noise $x_0 \sim p_0$ and data $x_1 \sim p_1$:
$$x_t = (1 - t) x_0 + t x_1, \quad u_t(x_t | x_0, x_1) = \frac{d x_t}{dt} = x_1 - x_0$$
The model optimizes:
$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t, x_0, x_1} \left[ \| v_\theta(x_t, t) - (x_1 - x_0) \|^2 \right]$$

### Consistency Boundary Parameterization
Song et al. enforce $f_\theta(x, \epsilon) = x$ via analytical scaling coefficients:
$$f_\theta(x, t) = c_{\text{skip}}(t) x + c_{\text{out}}(t) F_\theta(x, t)$$
where:
$$c_{\text{skip}}(t) = \frac{\sigma_{\text{data}}^2}{(t - \epsilon)^2 + \sigma_{\text{data}}^2}, \quad c_{\text{out}}(t) = \frac{\sigma_{\text{data}}(t - \epsilon)}{\sqrt{\sigma_{\text{data}}^2 + t^2}}$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# Generate synthetic 2D two-moon or Gaussian endpoints
B = 64
dim = 2
x_0 = torch.randn(B, dim)  # Source Gaussian noise p_0
radius = 3.0 + 0.3 * torch.randn(B, 1)
theta = torch.rand(B, 1) * 2 * np.pi
x_1 = torch.cat([radius * torch.cos(theta), radius * torch.sin(theta)], dim=-1)  # Target ring distribution p_1

t = torch.rand(B, 1)
print(f"Dataset generated: x_0 shape={x_0.shape}, x_1 shape={x_1.shape}, t range=[{t.min():.2f}, {t.max():.2f}]")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def conditional_flow_matching(x_0: torch.Tensor, x_1: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Computes linear interpolation and constant target velocity."""
    x_t = (1.0 - t) * x_0 + t * x_1
    v_target = x_1 - x_0
    return x_t, v_target

class VelocityMLP(torch.nn.Module):
    """Time-conditioned vector field prediction network."""
    def __init__(self, dim: int = 2, hidden_dim: int = 64):
        super().__init__()
        self.w_t = torch.nn.Linear(1, hidden_dim)
        self.net = torch.nn.Sequential(
            torch.nn.Linear(dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Linear(hidden_dim, dim),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = F.silu(self.w_t(t))
        # Add modulation
        h = F.silu(self.net[0](x) + t_emb)
        h = self.net[3](self.net[2](h))
        return self.net[4](h)

def consistency_coefficients(t: torch.Tensor, epsilon: float = 0.002, sigma_data: float = 0.5):
    dt = t - epsilon
    c_skip = (sigma_data ** 2) / (dt ** 2 + sigma_data ** 2)
    c_out = (sigma_data * dt) / torch.sqrt(sigma_data ** 2 + t ** 2)
    return c_skip, c_out

def consistency_forward(x: torch.Tensor, t: torch.Tensor, f_model: torch.nn.Module, epsilon: float = 0.002, sigma_data: float = 0.5):
    c_skip, c_out = consistency_coefficients(t, epsilon, sigma_data)
    return c_skip * x + c_out * f_model(x, t)

model = VelocityMLP(dim=2, hidden_dim=64)
x_t, v_target = conditional_flow_matching(x_0, x_1, t)
v_pred = model(x_t, t)
print(f"CFM forward complete: v_pred shape={v_pred.shape}, target shape={v_target.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: At t=0, x_t == x_0; at t=1, x_t == x_1
t_zero = torch.zeros(B, 1)
t_one = torch.ones(B, 1)
x_at_0, _ = conditional_flow_matching(x_0, x_1, t_zero)
x_at_1, _ = conditional_flow_matching(x_0, x_1, t_one)
assert torch.allclose(x_at_0, x_0, atol=1e-6), "Boundary invariant failed at t=0"
assert torch.allclose(x_at_1, x_1, atol=1e-6), "Boundary invariant failed at t=1"

# Invariant 2: Target velocity is strictly independent of t in straight CFM
_, v_target_0 = conditional_flow_matching(x_0, x_1, t_zero)
_, v_target_1 = conditional_flow_matching(x_0, x_1, t_one)
assert torch.allclose(v_target_0, v_target_1, atol=1e-6), "Velocity target must be constant along straight trajectory"

# Invariant 3: Consistency model boundary condition: f_theta(x, epsilon) == x
eps = 0.002
t_eps = torch.full((B, 1), eps)
f_at_eps = consistency_forward(x_0, t_eps, model, epsilon=eps)
diff = torch.max(torch.abs(f_at_eps - x_0)).item()
assert diff < 1e-6, f"Consistency boundary condition violated: max error {diff:.2e}"
print(f"All mathematical invariants verified successfully! Consistency boundary error: {diff:.2e}")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark 100 Euler steps vs 100 consistency calls
start = time.perf_counter()
with torch.no_grad():
    x_curr = x_0.clone()
    dt = 1.0 / 50.0
    for step in range(50):
        t_step = torch.full((B, 1), step * dt)
        v = model(x_curr, t_step)
        x_curr = x_curr + dt * v
euler_time = time.perf_counter() - start

start = time.perf_counter()
with torch.no_grad():
    x_single_step = consistency_forward(x_0, torch.ones(B, 1), model, epsilon=eps)
consistency_time = time.perf_counter() - start

print(f"50-step Euler ODE integration: {euler_time*1000:.2f} ms")
print(f"Single-step Consistency evaluation: {consistency_time*1000:.2f} ms")
print(f"Speedup from single-step consistency: {euler_time / max(consistency_time, 1e-6):.1f}x")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Failing to clamp or parameterize boundary conditions in consistency training
def unconstrained_consistency_bad(x: torch.Tensor, t: torch.Tensor, model: torch.nn.Module):
    # Missing skip connections: neural net must learn exact identity at t=eps by brute force
    return model(x, t)

# At t=eps, unconstrained model predicts garbage before extensive training
bad_output = unconstrained_consistency_bad(x_0, t_eps, model)
bad_error = torch.mean((bad_output - x_0) ** 2).item()

# Corrected Implementation: Skip-connection parameterization guarantees identity exactly
good_output = consistency_forward(x_0, t_eps, model, epsilon=eps)
good_error = torch.mean((good_output - x_0) ** 2).item()

print(f"Pathological (unconstrained) boundary MSE: {bad_error:.4f} (Violates boundary theorem)")
print(f"Corrected (parameterized skip) boundary MSE: {good_error:.8f} (Exact mathematical identity)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Architectural Principles
1. **Straight Trajectories Eliminate Truncation**: Unlike diffusion models with curved SDE dynamics, straight flow matching minimizes the Wasserstein-2 transport cost and curvature, allowing 2- to 4-step sampling with negligible perceptual degradation.
2. **Boundary Enforcement by Construction**: The skip-connection formulation $c_{\text{skip}}(t) x + c_{\text{out}}(t) F_\theta(x, t)$ renders the identity map at $t = \epsilon$ algebraically invariant to neural network weights, preventing boundary error accumulation.
3. **Reflow as Disentanglement**: Running iterative reflow turns intersecting ODE trajectories into parallel non-intersecting highways, directly overcoming multi-modal velocity averaging.